In [ ]:
RUN_ID = "20261106-132659" #this is with the force monoton increase
#RUN_ID = "20261106-133840" #this is without the force monoton increase
#RUN_ID = "20261107-211448" #latest run force monoton increase


In [ ]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.text import Text
from matplotlib.patches import Patch
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from scipy import stats as sstats
from catboost import CatBoostRegressor, CatBoostClassifier

REPO_ROOT = Path.cwd().parent
# make the shared code in src/ importable even when the package is not pip-installed
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from egypt_survey_ml.config import DatasetConfig
from egypt_survey_ml.dataset import prepare_dataset
from egypt_survey_ml.paths import DataPaths

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

from egypt_survey_ml.predict import load_members, predict_ensemble, describe, STAT_NAMES
from egypt_survey_ml.explain import plot_importance, plot_beeswarm



In [ ]:
# --- scenario settings -------------------------------------------------
# Every value below can be changed to test a different scenario. Defaults live in
# src/egypt_survey_ml/config.py (DatasetConfig); anything left out keeps its default
# (rng_seed=42, min_working_age=15, weeks_per_month=365.25/7/12).
# Column-selection workbook, separate from the command-area data_filename below.
# Sheet1: variable = column name, correl = 1 to include (0 to exclude), type = binary/
# categorical/numerical; remarks is ignored. Selected columns that are not in the data are
# skipped (same as the training notebooks). The saved models always get the columns they
# were trained on, so the mask only sets the default feature set of the dataset.


MASK_PATH = (
    Path.home()
    / "OneDrive - Stichting Deltares"
    / "PhD/Egypt_ERF_data/Data/variable_selection.xlsx"
)

cfg = DatasetConfig(
    include_empstab=True,  # see design-decisions table above: circular under the zero-inclusive regime
    include_hrswk=True,  # team decision (Vanni's email) excludes hours for the female model;
    # the audit branch's train-catboost_raw*.ipynb notebooks all keep it
    # for female too. Toggle this to test how much of their higher
    # reported R2 comes from that one feature, holding everything else
    # (repeated eval, intensive-margin decomposition, wage formula, cap)
    # fixed at our more rigorous setup.
    zero_handling="drop_random",  # one of: "none", "drop_random", "lognormal_synthetic"
    predict_log_wage=False,
    include_ribasim=True,
    wage_perturbation=False,
    wage_perturbation_sigma=0.03,
    drop_random_frac=1.0,  # only used when zero_handling == "drop_random"
    male_outlier_cap=10000,
    female_outlier_cap=np.inf,
    data_filename="Data/data_correlation.xlsx",
    #male_extra_columns=["water_supply", "water_productivity"],
    male_exclude_columns = [
    "female_violence_sexual",
    "female_violence_emotional",
    "female_violence_physical",
    "female_child_married",
    "female_circumcised",
    "female_healthcare_decision",
    "female_contraceptives_decision",
    "female_labour_participation_rate",
    "female_unemployment_rate",
]

)

In [ ]:
# Loads the survey, builds household wage context, makes the sex-stratified train/test split and the
# wage target, attaches governorates, and loads the command-area data and feature mask
# (see src/egypt_survey_ml/dataset.py).
_ONEDRIVE = "OneDrive - Stichting Deltares"
_EGYPT_DIR = "PhD/Egypt_ERF_data"
_LFS_DIR = "Data/hh_data"

results_dir = Path.home() / _ONEDRIVE / _EGYPT_DIR / "Results"
results_dir.mkdir(parents=True, exist_ok=True)


paths = DataPaths.from_onedrive(
    onedrive_dir=_ONEDRIVE,
    egypt_dir=_EGYPT_DIR,
    lfs_dir=_LFS_DIR,
    data_filename=cfg.data_filename,
)
ds = prepare_dataset(cfg, paths=paths, mask_path=MASK_PATH)

train_pool = ds.train_pool
test_pool = ds.test_pool
survey_columns = ds.survey_columns
excel_df = ds.excel_df
EXCEL_PATH = ds.paths.excel_path

# dataset-bound helpers used by the cells below
assign_command_units = ds.assign_command_units
build_feature_frame = ds.build_feature_frame
apply_zero_handling = ds.apply_zero_handling

In [ ]:
upper_egypt_govs = ['Aswan', 'Luxor', 'Quena', 'Suhag', 'Assyout', 'Menia', 'Beni-Suef', 'Fayoum', 'Cairo', 'Giza']

In [ ]:
MODEL_DIR = REPO_ROOT / 'models/hp-tuned'
SEEDS = [42]

classifier_dir = MODEL_DIR / f"classifiers-{RUN_ID}"
regressor_dir = MODEL_DIR / f"regressors-{RUN_ID}"

male_members = load_members(classifier_dir, regressor_dir, "male", seeds=SEEDS)
female_members = load_members(classifier_dir, regressor_dir, "female", seeds=SEEDS)

In [ ]:
# One example draw (seed 0) of the training frame, for exploring the data in the cells below
from egypt_survey_ml.dataset import TARGET_COLS

train_geo = ds.assign_command_units(train_pool, seed=0)
train_frame, _, _ = ds.build_feature_frame(train_geo, sex_code=2)
feature_cols = [c for c in train_frame.columns if c not in TARGET_COLS]

In [ ]:
# Every member predicts on the command-unit draw it was trained on (its seed).
# Each model picks its own feature columns, so the with/without RIBASIM models need no extra arguments.

# Specify 'governorates' to only predict for certain governorates.
pred_male = predict_ensemble(ds, male_members, 'male')#, governorates=upper_egypt_govs)
pred_female = predict_ensemble(ds, female_members, 'female')#, governorates=upper_egypt_govs)


In [ ]:
# female: hard classification at P(paid) >= 0.33; male: predicted wage weighted by P(paid)
r2_scores_female = pred_female.r2_per_member()#threshold=0.33
r2_scores_male = pred_male.r2_per_member()

r2_summary = pd.DataFrame({
    "metric": STAT_NAMES,
    "r2_female": describe(r2_scores_female),
    "r2_male": describe(r2_scores_male),
})

r2_summary
# r2_summary.to_csv('r2_summary_ribasim_with_hrs.csv')

In [ ]:
wage_dict = {}
scenario_predictions = {}
for ribasim_run in ["30", "40", "50", "60", "70", "80", "90"]:
    ribasim_excel_df = pd.read_excel(EXCEL_PATH.parent / "ribasim_data" / f"data_correlation_{ribasim_run}.xlsx", sheet_name="command_unit")
    scenario_predictions[ribasim_run] = {
        'female': predict_ensemble(ds, female_members, 'female', excel_df=ribasim_excel_df),
        'male': predict_ensemble(ds, male_members, 'male', excel_df=ribasim_excel_df),
    }

    for sex in ['female', 'male']:
        pred = scenario_predictions[ribasim_run][sex]
        wage_dict[f'{sex}_{ribasim_run}'] = describe(pred.final_wage(aggregate="median"), weights=None)

wage_df = pd.DataFrame(wage_dict, index=STAT_NAMES).sort_index(axis=1)

PERCENTILES = ["mean"]
X_VALUES = [30, 40, 50, 60, 70, 80, 90]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for ax, gender in zip(axes, ["female", "male"]):
    columns = [f"{gender}_{x}" for x in X_VALUES]

    for percentile in PERCENTILES:
        wages = wage_df.loc[percentile, columns]
        ax.plot(X_VALUES, wages, marker="o", label=percentile)

    ax.set_title(gender.capitalize())
    ax.set_xlabel("Scenario value")
    ax.set_xticks(X_VALUES)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(title="Percentile")

axes[0].set_ylabel("Wage")
plt.tight_layout()
plt.show()

wage_df



In [ ]:
runs = ["30", "40", "50", "60", "70", "80", "90"]
areas = pd.Index(ds.excel_df["area"].dropna().unique(), name="area")
results = []

for sex in ["female", "male"]:
    base = scenario_predictions["70"][sex]
    n_draws, n_workers = base.command_unit.shape
    draws = pd.DataFrame({"draw": np.repeat(np.arange(n_draws), n_workers), "area": base.command_unit.ravel()})

    # Baseline P5 among positive wages, separately for each command area and draw.
    draws["wage_70"] = np.tile(base.final_wage(aggregate="median"), n_draws)
    draws["positive_wage_70"] = draws["wage_70"].where(draws["wage_70"] > 0)
    draws["p5_70"] = draws.groupby(["draw", "area"])["positive_wage_70"].transform("quantile", q=0.05)

    # Working hours are fixed across scenarios.
    frame = ds.realization(sex, int(base.seeds[0]), test_data=True, zero_handling="none").frame
    draws["weekly_hours"] = np.tile(pd.to_numeric(frame["hrswk"], errors="coerce"), n_draws)

    for run in runs:
        pred = scenario_predictions[run][sex]
        if not np.array_equal(pred.seeds, base.seeds) or not pd.DataFrame(pred.command_unit).equals(pd.DataFrame(base.command_unit)):
            raise ValueError(f"Command-area assignments differ between scenarios 70 and {run} for {sex}.")

        wage = pd.Series(np.tile(pred.final_wage(aggregate="median"), n_draws))
        valid = wage.notna() & draws["p5_70"].notna()
        draws["pct_above_p5"] = (100 * wage.gt(draws["p5_70"])).where(valid)

        per_draw = draws.groupby(["draw", "area"])[["pct_above_p5", "p5_70", "weekly_hours"]].mean()
        by_area = per_draw.groupby("area").mean().reindex(areas).reset_index()
        results.append(by_area.assign(sex=sex, scenario=int(run)))

map_df = pd.concat(results, ignore_index=True)[["area", "sex", "scenario", "pct_above_p5", "p5_70", "weekly_hours"]]
map_df.to_csv(results_dir / 'unemployment_rate.csv')

map_df


In [ ]:
'''nonzero_wage_dict = {}
for ribasim_run, preds in scenario_predictions.items():
    for sex in ['female', 'male']:
        final_wage = preds[sex].final_wage(aggregate="median")
        nonzero_wage_dict[f'{sex}_{ribasim_run}'] = describe(final_wage[final_wage > 1000])

nonzero_wage_df = pd.DataFrame(nonzero_wage_dict, index=STAT_NAMES).sort_index(axis=1)

PERCENTILES = ["mean"]
X_VALUES = [30, 40, 50, 60, 70, 80, 90]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for ax, gender in zip(axes, ["female", "male"]):
    columns = [f"{gender}_{x}" for x in X_VALUES]

    for percentile in PERCENTILES:
        nonzero_wage = nonzero_wage_df.loc[percentile, columns]
        ax.plot(X_VALUES, nonzero_wage, marker="o", label=percentile)

    ax.set_title(gender.capitalize())
    ax.set_xlabel("Scenario value")
    ax.set_xticks(X_VALUES)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(title="Percentile")

axes[0].set_ylabel("Non-ZeroWage")
plt.tight_layout()
plt.show()

nonzero_wage_df
'''

In [ ]:

# Mean predicted wage per command unit: survey-weighted mean within each draw, then averaged over the draws.
# A unit without respondents in every draw is NaN.
wage_by_command_unit = pd.DataFrame({
    sex: pred.mean_by_command_unit(units=ds.excel_df["area"])["wage"]
    for sex, pred in [("female", pred_female), ("male", pred_male)]
})

wage_by_command_unit

wage_by_command_unit_dict = {}
for ribasim_run, preds in scenario_predictions.items():
    print(f"Ribasim run: {ribasim_run}")
    for sex, pred in preds.items():
        wage_by_command_unit_dict[f"{sex}_{ribasim_run}"] = pred.mean_by_command_unit(units=ds.excel_df["area"])["wage"]

wage_by_command_unit_df = pd.DataFrame(wage_by_command_unit_dict, index=ds.excel_df["area"])
wage_by_command_unit_df.to_csv(results_dir / 'wage_by_command_unit.csv')


In [ ]:
# Feature importance of the combined model (classifier x regressor, all draws): SHAP values of both models are
# converted to EGP, combined so they add up to P(paid) * wage, and averaged over the ensemble members.
# Takes roughly 12 minutes for both sexes (SHAP for every member).
from egypt_survey_ml.explain import combined_shap, plot_importance, plot_beeswarm

FIGURE_DIR = REPO_ROOT / 'figures'
FIGURE_DIR.mkdir(exist_ok=True)

shap_male = combined_shap(ds, male_members, 'male')
shap_female = combined_shap(ds, female_members, 'female')

importance = pd.DataFrame({
    'male': shap_male.importance(weighted=True),
    'female': shap_female.importance(weighted=True),
}).sort_values('male', ascending=False)

for sex, res in [('male', shap_male), ('female', shap_female)]:
    plot_importance(res.importance(weighted=True), top=20, title=f'Feature importance, combined model ({sex})')
    # plt.savefig(FIGURE_DIR / f'shap-importance-combined-{sex}.png')
    plt.show()

    # beeswarm: x = effect on the predicted wage (EGP), colour = feature value (grey = categorical)
    plot_beeswarm(res, top=20, title=f'Combined model ({sex})', show=False)
    # plt.savefig(FIGURE_DIR / f'shap-beeswarm-combined-{sex}.png', bbox_inches='tight')
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.container import BarContainer
from matplotlib.text import Text

feature_labels = {
    # Socio-economic indicators
    "empstab": "Employment stability",
    "hrswk": "Weekly working hours",
    "rel": "Relationship to household head",
    "tenure": "Job tenure",
    "age": "Age",
    "occ": "Occupation",
    "household_size": "Household size",
    "mart_d": "Marital status",
    "yeduc": "Years of education",
    "hlthins": "Health insurance",
    "socsec": "Social security coverage",
    "p_poor_population": "Share of population in poverty",
    "gdp_pc": "GDP per capita",
    "rwi_median": "Median relative wealth index",
    # Governorate-level gender indicators
    "female_contraceptives_decision": "Women's contraceptive decision-making",
    "female_child_married": "Child marriage prevalence",
    "female_circumcised": "Female genital mutilation prevalence",
    "female_violence_physical": "Physical violence against women",
    "female_violence_sexual": "Sexual violence against women",
    "female_healthcare_decision": "Women's healthcare decision-making",
    "female_labour_participation_rate": "Female labour force participation",
    # Livestock indicators
    "chicken": "Chickens per rural resident",
    "cattle": "Cattle per rural resident",
    "goats": "Goats per rural resident",
    # Agricultural and water indicators
    "producer_price": "Agricultural productivity*",
    "salinity_production_Rice, paddy": "Rice production*",
    "salinity_production_Wheat": "Wheat production*",
    "salinity_production_Alfalfa for forage": "Alfalfa production*",
    "water_productivity": "Water productivity*",
    "fraction_small": "Share of small farms",
}

# Fonts and feature-group colours
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman"], "font.size": 14, "axes.titlesize": 16, "axes.labelsize": 14, "xtick.labelsize": 12, "ytick.labelsize": 13, "legend.fontsize": 11, "text.color": "black", "axes.labelcolor": "black", "xtick.color": "black", "ytick.color": "black", "pdf.fonttype": 42, "ps.fonttype": 42})
productivity_features = {"producer_price", "salinity_production_Rice, paddy", "salinity_production_Wheat", "salinity_production_Alfalfa for forage", "fraction_small", "chicken", "cattle", "goats"}
water_features = {"water_productivity"}
productivity_color, water_color, socioeconomic_color = "#145A32", "#154360", "#922B21"
feature_colors = {feature: (productivity_color if feature in productivity_features else water_color if feature in water_features else socioeconomic_color) for feature in feature_labels}
legend_handles = [Patch(facecolor=water_color, label="Water"), Patch(facecolor=productivity_color, label="Agriculture"), Patch(facecolor=socioeconomic_color, label="Socio-economic")]
scenario_note = "* Modelled values vary across irrigation-efficiency scenarios."

FIGURE_DIR = REPO_ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
importance = pd.DataFrame({"male": shap_male.importance(weighted=True), "female": shap_female.importance(weighted=True)}).drop(index="V", errors="ignore").sort_values("male", ascending=False)

for sex, res in [("male", shap_male), ("female", shap_female)]:
    # Feature importance: coloured bars and black text
    plot_importance(res.importance(weighted=True).drop(index="V", errors="ignore"), top=20, title=f" ")#Feature importance, combined model ({sex})
    ax = plt.gca()
    fig = ax.figure
    original_labels = [tick.get_text() for tick in ax.get_yticklabels()]
    tick_positions = ax.get_yticks()
    for container in ax.containers:
        if isinstance(container, BarContainer):
            for bar in container.patches:
                bar_center = bar.get_y() + bar.get_height() / 2
                nearest_tick = min(range(len(tick_positions)), key=lambda i: abs(tick_positions[i] - bar_center))
                bar.set_facecolor(feature_colors.get(original_labels[nearest_tick], "#666666"))
                bar.set_edgecolor("none")
    ax.set_yticks(tick_positions, labels=[feature_labels.get(feature, feature) for feature in original_labels])
    ax.set_xlabel(f"Mean absolute attribution to predicted {sex} wages (EGP)")
    ax.set_ylabel("")
    fig.legend(handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, 0.045), ncol=3, frameon=False)
    fig.text(0.5, 0.018, scenario_note, ha="center", va="bottom", fontsize=11, color="black")
    for text in fig.findobj(match=Text):
        text.set_fontfamily("Times New Roman")
        text.set_color("black")
    fig.tight_layout(rect=[0, 0.11, 1, 1])
    fig.savefig(FIGURE_DIR / f"shap-importance-combined-{sex}.png", dpi=300, bbox_inches="tight")
    plt.show()

    # Beeswarm: retain feature-value dot colours; V must be filtered within plot_beeswarm
    plot_beeswarm(res, top=20, title=f"Combined model ({sex})", show=False)
    ax = plt.gca()
    fig = ax.figure
    original_labels = [tick.get_text() for tick in ax.get_yticklabels()]
    ax.set_yticks(ax.get_yticks(), labels=[feature_labels.get(feature, feature) for feature in original_labels])
    ax.set_xlabel(f"Attribution to predicted {sex} wages (EGP)")
    ax.set_ylabel("")
    # No group legend here because neither dots nor labels encode feature groups.
    fig.text(0.5, 0.018, scenario_note, ha="center", va="bottom", fontsize=11, color="black")
    for text in fig.findobj(match=Text):
        text.set_fontfamily("Times New Roman")
        text.set_color("black")
    fig.tight_layout(rect=[0, 0.065, 1, 1])
    fig.savefig(FIGURE_DIR / f"shap-beeswarm-combined-{sex}.png", dpi=300, bbox_inches="tight")
    plt.show()

# Importance table with readable labels
importance.rename(index=feature_labels)

In [ ]:
wage_df.to_csv(results_dir / "wages_predictions.csv")
#nonzero_wage_df.to_csv(results_dir / "nonzero_wages_predictions.csv")

In [ ]:
prediction_paid_male = pred_male.predicted_paid(threshold=0.70)
prediction_final_male = pred_male.final_wage()
true_paid_male = pred_male.true_paid
true_income_male = pred_male.true_income

y_pred = prediction_paid_male.astype(float)
y_true = true_paid_male

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(cm)
disp.plot(cmap="Blues")

plt.show()

In [ ]:
prediction_paid_female = pred_female.predicted_paid(threshold=0.33)
prediction_final_female = pred_female.final_wage()
true_paid_female = pred_female.true_paid
true_income_female = pred_female.true_income

y_pred = prediction_paid_female.astype(float)
y_true = true_paid_female

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(cm)
disp.plot(cmap="Blues")

plt.show()

In [ ]:
prediction_paid_male = pred_male.predicted_paid()
prediction_final_male = pred_male.final_wage()

y_pred = prediction_paid_male.astype(float)
y_true = true_paid_male

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(cm)
disp.plot(cmap="Blues")

plt.show()

In [ ]:
prediction_paid_female = pred_female.predicted_paid(threshold=0.33)
prediction_final_female = pred_female.final_wage()

y_pred = prediction_paid_female.astype(float)
y_true = true_paid_female

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(cm)
disp.plot(cmap="Blues")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
y_pred_male = prediction_final_male
y_true_male = true_income_male
y_pred_nr_male = prediction_final_male

y_pred_male = y_pred_male[y_true_male > 0]
y_pred_nr_male = y_pred_nr_male[y_true_male > 0]
y_true_male = y_true_male[y_true_male > 0]

ax.scatter(y_true_male, y_pred_male, alpha=0.4, s=15)
ax.scatter(y_true_male, y_pred_nr_male, alpha=0.4, s=15, color='orange')
lims = [0.95 * min(y_true_male.min(), y_pred_male.min()), max(y_true_male.max(), y_pred_male.max()) * 1.05]
ax.plot(lims, lims, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Actual wage (EGP)")
ax.set_title(f"male (n={len(y_true)})")
ax.set_ylabel("Predicted wage (EGP)")
plt.tight_layout()
plt.show()

metrics = {
    "n": len(y_true_male),
    "mae": mean_absolute_error(y_true_male, y_pred_male),
    "r2": r2_score(y_true_male, y_pred_male),
    "spearman": sstats.spearmanr(y_true_male, y_pred_male).correlation,
}
print(metrics)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
y_pred_female = prediction_final_female
y_pred_nr_female = prediction_final_female
y_true_female = true_income_female

y_pred_female = y_pred_female[y_true_female > 0]
y_pred_nr_female = y_pred_nr_female[y_true_female > 0]
y_true_female = y_true_female[y_true_female > 0]

ax.scatter(y_true_female, y_pred_female, alpha=0.4, s=15, label='With RIBASIM')
ax.scatter(y_true_female, y_pred_nr_female, alpha=0.4, s=15, color='orange', label='Without RIBASIM')
lims = [0.95 * min(y_true_female.min(), y_pred_female.min()), max(y_true_female.max(), y_pred_female.max()) * 1.05]
ax.plot(lims, lims, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Actual wage (EGP)")
ax.set_title(f"female (n={len(y_true)})")
ax.set_ylabel("Predicted wage (EGP)")
ax.legend()
plt.tight_layout()
plt.show()

metrics = {
    "n": len(y_true_female),
    "mae": mean_absolute_error(y_true_female, y_pred_female),
    "r2": r2_score(y_true_female, y_pred_female),
    "spearman": sstats.spearmanr(y_true_female, y_pred_female).correlation,
}
metrics_nr = {
    "n": len(y_true_female),
    "mae": mean_absolute_error(y_true_female, y_pred_nr_female),
    "r2": r2_score(y_true_female, y_pred_nr_female),
    "spearman": sstats.spearmanr(y_true_female, y_pred_nr_female).correlation,
}
print(metrics)
print(metrics_nr)

In [ ]:
FIGURE_DIR = results_dir / "figure"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
y_pred_male = prediction_final_male
y_true_male = true_income_male
y_pred_nr_male = prediction_final_male

y_pred_male = y_pred_male[y_true_male > 0]
y_pred_nr_male = y_pred_nr_male[y_true_male > 0]
y_true_male = y_true_male[y_true_male > 0]

y_pred_female = prediction_final_female
y_pred_nr_female = prediction_final_female
y_true_female = true_income_female

y_pred_female = y_pred_female[y_true_female > 0]
y_pred_nr_female = y_pred_nr_female[y_true_female > 0]
y_true_female = y_true_female[y_true_female > 0]

fig, ax = plt.subplots(figsize=(16, 8))

data = [
    y_true_male,
    y_pred_nr_male,
    y_pred_male,
    y_true_female,
    y_pred_nr_female,
    y_pred_female,
]

labels = [
    "Truth\nMale",
    "No Ribasim\nMale",
    "Ribasim\nMale",
    "Truth\nFemale",
    "No Ribasim\nFemale",
    "Ribasim\nFemale",
]

bp = ax.boxplot(
    data,
    patch_artist=True,
    showmeans=True,
    showfliers=False,
)

colors = [
    "#0072B2", "#0072B2",
    "#0072B2", "#E69F00",
    "#E69F00", "#E69F00",
]

for box, color in zip(bp["boxes"], colors):
    box.set_facecolor(color)
    box.set_alpha(0.8)

ax.set_xticklabels(labels)

ax.set_ylabel("Income")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
plt.savefig(FIGURE_DIR / 'baseline-box-plots-with-hrs-22.png')

In [ ]:
y_pred_male = prediction_final_male
y_true_male = true_income_male
y_pred_nr_male = prediction_final_male

y_pred_female = prediction_final_female
y_pred_nr_female = prediction_final_female
y_true_female = true_income_female

fig, ax = plt.subplots(figsize=(16, 8))

data = [
    y_true_male,
    y_pred_nr_male,
    y_pred_male,
    y_true_female,
    y_pred_nr_female,
    y_pred_female,
]

positions = [1, 2, 3, 5, 6, 7]

vp = ax.violinplot(
    data,
    positions=positions,
    widths=0.8,
    showmeans=True,
    showmedians=True,
)

colors = [
    "#0072B2", "#0072B2",  # Truth
    "#0072B2", "#E69F00",  # Ribasim
    "#E69F00", "#E69F00",  # No Ribasim
]

for body, color in zip(vp["bodies"], colors):
    body.set_facecolor(color)
    body.set_edgecolor("black")
    body.set_alpha(0.7)

ax.set_xticks(positions)
ax.set_xticklabels([
    "Truth\nMale",
    "No Ribasim\nMale",
    "Ribasim\nMale",
    "Truth\nFemale",
    "No Ribasim\nFemale",
    "Ribasim\nFemale",
])

ax.set_ylabel("Income")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
# plt.show()
plt.savefig(FIGURE_DIR / 'baseline-violin-plots-with-hrs-22.png')